In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenRouter(
    model="deepseek/deepseek-v4.1-flash",
    temperature=0.7,
    max_tokens=10000
)

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):
    prompt = f"generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content
    return {"joke": response}
 
def generate_explanation(state: JokeState):
    prompt = f"write an explanation for this joke: {state['joke']}"
    response = llm.invoke(prompt).content
    return {"explanation": response}

In [8]:
checkpointer = MemorySaver()

graph = StateGraph(JokeState)
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

workflow = graph.compile(checkpointer=checkpointer)


In [ ]:
config = {"configurable": {"thread_id": "1"}}
 
result = workflow.invoke({"topic": "pizza"}, config=config)
print(result) # result thì luôn trả về state 


{'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it was feeling a little crusty. 🍕', 'explanation': 'This is a pun-based joke. The word **“crusty”** has two meanings:\n\n1. **Literally:** having a crust — which pizza naturally has.  \n2. **Colloquially:** feeling rough, grimy, irritable, or unwell — like when someone says they feel “a little crusty.”\n\nSo the pizza goes to the doctor because it “feels crusty,” which sounds like a medical complaint. But for a pizza, being crusty is completely normal. The humor comes from treating the pizza like a patient and mixing up the two meanings of “crusty.” 🍕'}


In [ ]:
workflow.get_state(config) #checkpoint laster new

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it was feeling a little crusty. 🍕', 'explanation': 'This is a pun-based joke. The word **“crusty”** has two meanings:\n\n1. **Literally:** having a crust — which pizza naturally has.  \n2. **Colloquially:** feeling rough, grimy, irritable, or unwell — like when someone says they feel “a little crusty.”\n\nSo the pizza goes to the doctor because it “feels crusty,” which sounds like a medical complaint. But for a pizza, being crusty is completely normal. The humor comes from treating the pizza like a patient and mixing up the two meanings of “crusty.” 🍕'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bae76-6aed-6d28-8002-1cd10e7818fc'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-28T02:50:44.814352+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bae76-22a1-614a-8

In [ ]:
list(workflow.get_state_history(config)) # get all checkpoint 

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\n\nBecause it was feeling a little crusty. 🍕', 'explanation': 'This is a pun-based joke. The word **“crusty”** has two meanings:\n\n1. **Literally:** having a crust — which pizza naturally has.  \n2. **Colloquially:** feeling rough, grimy, irritable, or unwell — like when someone says they feel “a little crusty.”\n\nSo the pizza goes to the doctor because it “feels crusty,” which sounds like a medical complaint. But for a pizza, being crusty is completely normal. The humor comes from treating the pizza like a patient and mixing up the two meanings of “crusty.” 🍕'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bae76-6aed-6d28-8002-1cd10e7818fc'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-28T02:50:44.814352+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bae76-22a1-614a-

In [ ]:
workflow.get_state_history(config) #generator 

<generator object Pregel.get_state_history at 0x0000024BE5C8EF00>

In [15]:
for snap in workflow.get_state_history(config):
    print(snap.next, snap.config["configurable"]["checkpoint_id"])


() 1f1bae76-6aed-6d28-8002-1cd10e7818fc
('generate_explanation',) 1f1bae76-22a1-614a-8001-4763840113bf
('generate_joke',) 1f1bae75-d69a-6a61-8000-24b332d1483f
('__start__',) 1f1bae75-d695-6be8-bfff-a172295eba4d
